In [ ]:
!apt-get update -qq && apt-get install -y ffmpeg -qq
!pip install -q ultralytics torchaudio pandas opencv-python scenedetect tqdm
!wget -q -O yolov8n-face.pt https://huggingface.co/junjiang/GestureFace/resolve/main/yolov8n-face.pt

In [ ]:
!ffmpeg -hide_banner -encoders | grep nvenc

In [ ]:
!ffmpeg -hide_banner -hwaccels

In [ ]:
%%writefile worker_module.py
import os, cv2, torch, subprocess, fcntl
import numpy as np
import pandas as pd
from ultralytics import YOLO
from tqdm import tqdm

VAD_MIN_DURATION = 2.0
VAD_MAX_DURATION = 12.0
WINDOW_SIZE = 5.0
WINDOW_STEP = 4.0
TAIL_MIN_DURATION = 2.5
YOLO_PASS_RATIO = 0.7
MIN_SPEECH_RATIO = 0.4
HISTOGRAM_CORRELATION_THRESHOLD = 0.6
AUDIO_BITRATE = "128k"

# --- Cấu hình encode/decode ---
USE_NVENC          = True    # đặt False nếu không có h264_nvenc
USE_HWACCEL_DECODE = True    # đặt False nếu 'ffmpeg -hwaccels' không có 'cuda'
NVENC_CQ     = 23
NVENC_PRESET = "p4"
X264_CRF     = 18            # dùng khi USE_NVENC=False
X264_PRESET  = "veryfast"
YOLO_BATCH   = 256           # chia nhỏ predict để tránh OOM


def extract_audio(video_path, out_wav):
    result = subprocess.run([
        'ffmpeg', '-y', '-threads', '2', '-i', video_path,
        '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1',
        '-loglevel', 'error', out_wav
    ], capture_output=True)
    return os.path.exists(out_wav) and os.path.getsize(out_wav) > 0


def get_all_frames_1fps(video_path, gpu_id):
    hw = ['-hwaccel', 'cuda', '-hwaccel_device', str(gpu_id)] if USE_HWACCEL_DECODE else ['-threads', '2']
    cmd = [
        'ffmpeg', '-y', *hw,
        '-i', video_path,
        '-vf', 'fps=1,scale=320:320',
        '-f', 'rawvideo', '-pix_fmt', 'bgr24',
        '-loglevel', 'error', 'pipe:1'
    ]
    result = subprocess.run(cmd, capture_output=True)
    raw = result.stdout
    frame_size = 320 * 320 * 3
    n_frames = len(raw) // frame_size
    frames = []
    for i in range(n_frames):
        fb = raw[i * frame_size:(i + 1) * frame_size]
        frames.append(np.frombuffer(fb, dtype=np.uint8).reshape(320, 320, 3).copy())
    return frames


def build_chunks(start_sec, end_sec):
    duration = end_sec - start_sec
    chunks = []
    if duration < VAD_MIN_DURATION:
        return chunks
    if duration <= VAD_MAX_DURATION:
        chunks.append((start_sec, end_sec))
    else:
        s = start_sec
        while s + WINDOW_SIZE <= end_sec:
            chunks.append((s, s + WINDOW_SIZE))
            s += WINDOW_STEP
        tail_dur = end_sec - s
        if tail_dur >= TAIL_MIN_DURATION:
            chunks.append((s, end_sec))
    return chunks


def check_scene_cut(frames):
    prev_hist = None
    for frame in frames:
        hist = cv2.calcHist([frame], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256])
        cv2.normalize(hist, hist, alpha=0, beta=1, norm_type=cv2.NORM_MINMAX)
        if prev_hist is not None:
            sim = cv2.compareHist(prev_hist, hist, cv2.HISTCMP_CORREL)
            if sim < HISTOGRAM_CORRELATION_THRESHOLD:
                return True
        prev_hist = hist
    return False


def speech_ratio_in_window(c_start, c_end, vad_segments):
    window = c_end - c_start
    if window <= 0:
        return 0.0
    overlap = 0.0
    for seg in vad_segments:
        lo = max(c_start, seg['start'] / 16000.0)
        hi = min(c_end, seg['end'] / 16000.0)
        if hi > lo:
            overlap += hi - lo
    return overlap / window


def cut_clip(video_path, clip_path, c_start, c_end, gpu_id):
    hw = ['-hwaccel', 'cuda', '-hwaccel_device', str(gpu_id)] if USE_HWACCEL_DECODE else []
    if USE_NVENC:
        vcodec = ['-c:v', 'h264_nvenc', '-preset', NVENC_PRESET,
                  '-rc', 'vbr', '-cq', str(NVENC_CQ), '-gpu', str(gpu_id),
                  '-pix_fmt', 'yuv420p']
    else:
        vcodec = ['-c:v', 'libx264', '-crf', str(X264_CRF), '-preset', X264_PRESET]
    cmd = [
        'ffmpeg', '-y', *hw,
        '-ss', str(c_start), '-i', video_path,
        '-t', str(c_end - c_start),
        *vcodec,
        '-c:a', 'aac', '-b:a', AUDIO_BITRATE,
        '-loglevel', 'error', clip_path
    ]
    subprocess.run(cmd, capture_output=True)
    return os.path.exists(clip_path) and os.path.getsize(clip_path) > 0


def append_to_csv_locked(filepath, data_list):
    if not data_list:
        return
    df = pd.DataFrame(data_list)
    with open(filepath, 'a') as f:
        fcntl.flock(f, fcntl.LOCK_EX)
        try:
            df.to_csv(f, header=(os.path.getsize(filepath) == 0), index=False)
        finally:
            fcntl.flock(f, fcntl.LOCK_UN)


def detect_faces_batched(face_model, frames, gpu_id, batch=YOLO_BATCH):
    flags = []
    for i in range(0, len(frames), batch):
        res = face_model.predict(frames[i:i + batch], verbose=False,
                                  imgsz=320, half=True, device=gpu_id, stream=False)
        flags.extend(len(r.boxes) > 0 for r in res)
        del res
    return flags


def worker_process(worker_id, gpu_id, videos, dataset_dir, output_dir, clip_log_path, reject_log_path):
    device = torch.device(f'cuda:{gpu_id}' if torch.cuda.is_available() else 'cpu')

    face_model = YOLO('yolov8n-face.pt')
    face_model.model = face_model.model.to(device)
    face_model.model.eval()

    vad_model, utils = torch.hub.load(
        repo_or_dir='snakers4/silero-vad', model='silero_vad',
        force_reload=False, trust_repo=True
    )
    vad_model = vad_model.to(device).eval()
    get_speech_timestamps, _, read_audio, _, _ = utils

    for filename in tqdm(videos, position=worker_id, desc=f"W{worker_id}(GPU{gpu_id})"):
        valid_clips = []
        reject_log = []

        video_path = os.path.join(dataset_dir, filename)
        video_id = os.path.splitext(filename)[0]
        temp_audio = f"/tmp/{video_id}_src_{worker_id}.wav"

        if not extract_audio(video_path, temp_audio):
            reject_log.append({'video': video_id, 'reason': 'audio_extract_failed'})
            append_to_csv_locked(reject_log_path, reject_log)
            continue

        try:
            wav = read_audio(temp_audio).to(device)
            vad_segments = get_speech_timestamps(wav, vad_model, sampling_rate=16000)
        except Exception:
            reject_log.append({'video': video_id, 'reason': 'vad_failed'})
            if os.path.exists(temp_audio): os.remove(temp_audio)
            append_to_csv_locked(reject_log_path, reject_log)
            continue

        if not vad_segments:
            reject_log.append({'video': video_id, 'reason': 'no_speech_detected'})
            if os.path.exists(temp_audio): os.remove(temp_audio)
            append_to_csv_locked(reject_log_path, reject_log)
            continue

        all_frames = get_all_frames_1fps(video_path, gpu_id)
        if not all_frames:
            reject_log.append({'video': video_id, 'reason': 'video_decode_failed'})
            if os.path.exists(temp_audio): os.remove(temp_audio)
            append_to_csv_locked(reject_log_path, reject_log)
            continue

        frame_has_face = detect_faces_batched(face_model, all_frames, gpu_id)

        clip_idx = 0
        for seg in vad_segments:
            seg_start = seg['start'] / 16000.0
            seg_end = seg['end'] / 16000.0
            if (seg_end - seg_start) < VAD_MIN_DURATION:
                reject_log.append({'video': video_id, 'start': round(seg_start, 3),
                                   'end': round(seg_end, 3), 'reason': 'segment_too_short'})
                continue

            for c_start, c_end in build_chunks(seg_start, seg_end):
                c_dur = c_end - c_start
                start_idx = int(c_start)
                end_idx = int(c_end)
                if end_idx == start_idx:
                    end_idx += 1

                chunk_frames = all_frames[start_idx:end_idx]
                if not chunk_frames:
                    reject_log.append({'video': video_id, 'start': round(c_start, 3),
                                       'end': round(c_end, 3), 'reason': 'frame_extract_failed'})
                    continue

                if check_scene_cut(chunk_frames):
                    reject_log.append({'video': video_id, 'start': round(c_start, 3),
                                       'end': round(c_end, 3), 'reason': 'scene_cut_detected'})
                    continue

                face_flags = frame_has_face[start_idx:end_idx]
                if not face_flags:
                    reject_log.append({'video': video_id, 'start': round(c_start, 3),
                                       'end': round(c_end, 3), 'reason': 'face_index_oob'})
                    continue
                face_ratio = sum(face_flags) / len(face_flags)
                if face_ratio < YOLO_PASS_RATIO:
                    reject_log.append({'video': video_id, 'start': round(c_start, 3),
                                       'end': round(c_end, 3), 'reason': 'face_density_low'})
                    continue

                speech_ratio = speech_ratio_in_window(c_start, c_end, vad_segments)
                if speech_ratio < MIN_SPEECH_RATIO:
                    reject_log.append({'video': video_id, 'start': round(c_start, 3),
                                       'end': round(c_end, 3), 'reason': 'speech_low'})
                    continue

                clip_name = f"{video_id}_clip{clip_idx:04d}_t{int(c_start):05d}.mp4"
                clip_path = os.path.join(output_dir, clip_name)
                if not cut_clip(video_path, clip_path, c_start, c_end, gpu_id):
                    reject_log.append({'video': video_id, 'start': round(c_start, 3),
                                       'end': round(c_end, 3), 'reason': 'ffmpeg_cut_failed'})
                    continue

                valid_clips.append({
                    'clip_id': clip_name.replace('.mp4', ''),
                    'source_video': video_id,
                    'start_time': round(c_start, 3),
                    'end_time': round(c_end, 3),
                    'duration': round(c_dur, 3),
                    'face_ratio': round(face_ratio, 3),
                    'speech_ratio': round(speech_ratio, 3),
                    'file_path': clip_path
                })
                clip_idx += 1

        if os.path.exists(temp_audio): os.remove(temp_audio)
        append_to_csv_locked(clip_log_path, valid_clips)
        append_to_csv_locked(reject_log_path, reject_log)

    return True

In [ ]:
import os, time, shutil, glob
import pandas as pd
import multiprocessing as mp
import torch
from worker_module import worker_process

TIER_NAME = "tier1"
DATASET_DIR = "/kaggle/input/datasets/xanhla/vn-av-df-data-tier-1/tier1"
INPUT_CSV = "/kaggle/input/datasets/xanhla/tier1-quality-gate-passed-csv/tier1_quality_gate_passed.csv"
WORKING_DIR = "/kaggle/working"
CLIPS_DIR = os.path.join(WORKING_DIR, "clips", TIER_NAME)
os.makedirs(CLIPS_DIR, exist_ok=True)

START_INDEX = 0
END_INDEX = 100
BATCH_TAG = f"{START_INDEX}_{END_INDEX}"

NUM_WORKERS = 2   # 1 worker / GPU. Đo xong rồi mới thử nâng lên 4.

CLIP_LOG_MAIN = os.path.join(WORKING_DIR, f"{TIER_NAME}_v3_clips_{BATCH_TAG}.csv")
REJECT_LOG_MAIN = os.path.join(WORKING_DIR, f"{TIER_NAME}_v3_rejects_{BATCH_TAG}.csv")

for f in glob.glob(os.path.join(WORKING_DIR, f"{TIER_NAME}_v3_*_{BATCH_TAG}_w*.csv")):
    os.remove(f)

if __name__ == '__main__':
    mp.set_start_method('spawn', force=True)
    torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad',
                   force_reload=False, trust_repo=True)

    num_gpus = max(torch.cuda.device_count(), 1)

    try:
        df = pd.read_csv(INPUT_CSV)
        passed_videos = df['filename'].tolist()
    except FileNotFoundError:
        passed_videos = []
    if not passed_videos:
        raise SystemExit("Khong co video")

    videos_to_process = passed_videos[START_INDEX:END_INDEX]
    shards = [videos_to_process[i::NUM_WORKERS] for i in range(NUM_WORKERS)]

    clip_logs   = [os.path.join(WORKING_DIR, f"{TIER_NAME}_v3_clips_{BATCH_TAG}_w{i}.csv")   for i in range(NUM_WORKERS)]
    reject_logs = [os.path.join(WORKING_DIR, f"{TIER_NAME}_v3_rejects_{BATCH_TAG}_w{i}.csv") for i in range(NUM_WORKERS)]

    t0 = time.time()
    with mp.Pool(processes=NUM_WORKERS) as pool:
        results = []
        for i in range(NUM_WORKERS):
            gpu_id = i % num_gpus
            results.append(pool.apply_async(
                worker_process,
                args=(i, gpu_id, shards[i], DATASET_DIR, CLIPS_DIR, clip_logs[i], reject_logs[i])
            ))
        for r in results:
            r.get()

    def merge_csvs(files, out_file):
        dfs = [pd.read_csv(f) for f in files if os.path.exists(f) and os.path.getsize(f) > 0]
        if dfs:
            pd.concat(dfs, ignore_index=True).to_csv(out_file, index=False)

    merge_csvs(clip_logs, CLIP_LOG_MAIN)
    merge_csvs(reject_logs, REJECT_LOG_MAIN)

    total_time = time.time() - t0
    try:    total_clips = len(pd.read_csv(CLIP_LOG_MAIN))
    except FileNotFoundError: total_clips = 0
    try:    total_rejects = len(pd.read_csv(REJECT_LOG_MAIN))
    except FileNotFoundError: total_rejects = 0

    print(f"Tổng: {total_clips} clips | {total_rejects} rejects")
    print(f"Thời gian: {total_time/60:.1f} phút")

    shutil.make_archive(os.path.join(WORKING_DIR, f"{TIER_NAME}_v3_clips_{BATCH_TAG}"), 'zip', CLIPS_DIR)
    print("Hoàn tất")